## 회원 CRUD - auth.users, user_details
- 삭제 : deleted_at에 날짜와 시간이 업데이트되면 소프트 삭제
- 조회시에는 deleted_at is not null로 조회될 수 있도록 한다.

In [4]:
import os
from supabase import create_client
from dotenv import load_dotenv
load_dotenv()

url = os.getenv("API_URL")
key = os.getenv("API_KEY")

supabase = create_client(url, key)
print(supabase)


In [ ]:

from datetime import datetime, timezone

class AuthUsers:

    def __init__(self,supabase):
        self.supabase = supabase
        self.user = None

        try:
            response = (
                self.supabase.auth.get_user()
            )
            
            self.user = response.user

        except Exception:
            print("가입된 회원 정보가 없습니다.")

    
    # [CREATE] 회원가입
    def sign_up(self, mail, pw):
        response = (
            self.supabase.auth.sign_up({
                "email":mail,
                "password":pw
            })
        )

        return response

    # [AUTH] 로그인
    def sign_in(self, mail, pw):
        response = (
            self.supabase.auth.sign_in_with_password({
                "email": mail,
                "password": pw
            })
        )

        self.user = response.user

        return response

    # [UPDATE] 회원 기본정보 수정 
    # auth.users에서 수정 가능한 이메일 또는 비밀번호 변경
    # 회원 상세 정보 수정과 별도 기능으로 구현
    def update(self, data):
        self.check_authority()

        allowed_fields = {"email","password"}
        invalid_fields = set(data.keys()) - allowed_fields

        if invalid_fields:
            raise ValueError ("해당 정보는 [회원 상세 정보]에서 수정할 수 있습니다.")

        response = (
            self.supabase.auth.update_user(data)
            )

        self.user = response.user

        return response

    # [DELETE] 회원 탈퇴
    # 실제 데이터를 삭제하지 않고 deleted_at을 기록하는 Soft Delete
    def resign(self):
        self.check_authority()
        response = (
            self.supabase.table("user_details")
            .update({
                "deleted_at": datetime.now(timezone.utc)
                .isoformat()
            })
            .eq("id", self.user.id)
            .is_("deleted_at", "NULL")
            .execute()
            )
        
        return response.data
        
        
    # [AUTH] 로그아웃
    def sign_out(self):
        self.check_authority()
        response = (
            self.supabase.auth.sign_out()
        )

        self.user = None

        return response

     # [READ] 현재 로그인한 회원 정보 조회
    def get_my_info(self):
        self.check_authority()
        response = (
            self.supabase.table("user_details")
                .select("*")
                .eq("id", user_id)
                .maybe_single()
                .execute()
        )
        if response.data:
            # auth.users의 이메일을 일부 마스킹하여 추가
            # email = self.user.email
            # name, domain = email.split("@")

            # masked_email = (
            #     name[:2]
            #     + "*" * max(len(name) - 2, 1)
            #     + "@"
            #     + domain
            # )

            # data["email"] = masked_email
            data = dict(response.data)
            data['email'] = self.user.email

            return data

        return None 


    def get_users(self, page = 1, limit = 20, *, search = None):
        pass

    def is_logged_in(self):
        if self.user: 
            return True

        return False

    def check_authority(self):
        if not self.is_logged_in():
            raise PermissionError("로그인이 필요합니다.")

    

        
    